In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import h5py

# Make repository modules importable when this notebook is launched from scripts/.
project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'metrics.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate the repository root containing metrics.py')
project_root_string = str(project_root)
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)

from matplotlib import pyplot as plt
from matplotlib import colors
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
from matplotlib.colors import Normalize
import cmcrameri.cm as cmc

%matplotlib inline

## Load all reconstructions

In [ ]:
# GT
with h5py.File("/mnt/localdisk/scratch/aileenluo/fig2/gt_filtered_lsqml/gt_filtered_lsqml_epoch300_para.hdf5", "r") as f:
    gt_lsqml = f["object"][0, 113:-114, 125:-125]
amp_gt_lsqml = np.abs(gt_lsqml)
ph_gt_lsqml = np.angle(gt_lsqml)

# Long exposure ePIE
with h5py.File("/mnt/localdisk/scratch/aileenluo/fig2/gt_filtered_epie/gt_filtered_epie_epoch3000_para.hdf5", "r") as f:
    gt_epie = f["object"][0, 113:-114, 125:-125]
amp_gt_epie = np.abs(gt_epie)
ph_gt_epie = np.angle(gt_epie)

# Flyscan LSQML
with h5py.File("/mnt/localdisk/scratch/aileenluo/fig2/scan478_lsqml/scan478_lsqml_jitter_epoch900_para.hdf5", "r") as f:
    fly_lsqml = f["object"][0, 154:-154, 154:-154]
amp_fly_lsqml = np.abs(fly_lsqml)
ph_fly_lsqml = np.angle(fly_lsqml)

# Flyscan ePIE
with h5py.File("/mnt/localdisk/scratch/aileenluo/fig2/scan478_epie/scan478_epie_poscorr_epoch3000_para.hdf5", "r") as f:
    fly_epie = f["object"][0, 154:-154, 154:-154]
amp_fly_epie = np.abs(fly_epie)
ph_fly_epie = np.angle(fly_epie)

# Flyscan DM
with h5py.File("/mnt/localdisk/scratch/aileenluo/fig2/scan478_dm/scan478_epoch20_para.hdf5", "r") as f:
    fly_dm = f["object"][0, 154:-154, 154:-154]
amp_fly_dm = np.abs(fly_dm)
ph_fly_dm = np.angle(fly_dm)

# Flyscan RAAR
with h5py.File("/mnt/localdisk/scratch/aileenluo/fig2/scan478_raar/scan478_epoch15_para.hdf5", "r") as f:
    fly_raar = f["object"][0, 154:-154, 154:-154]
amp_fly_raar = np.abs(fly_raar)
ph_fly_raar = np.angle(fly_raar)

# Flyscan rPIE
with h5py.File("/mnt/localdisk/scratch/aileenluo/fig2/scan478_rpie/scan478_epoch35_para.hdf5", "r") as f:
    fly_rpie = f["object"][0, 154:-154, 154:-154]
amp_fly_rpie = np.abs(fly_rpie)
ph_fly_rpie = np.angle(fly_rpie)

# Flyscan inference 43M fine-tuned
amp_43M_finetune = np.load("/mnt/localdisk/scratch/aileenluo/results/run164/pred_amp_object_scan478_crop104_datanorm.npy")
ph_43M_finetune = np.load("/mnt/localdisk/scratch/aileenluo/results/run164/pred_ph_object_scan478_crop104_datanorm.npy")

# Flyscan inference 200M fine-tuned
amp_200M_finetune = np.load("/mnt/localdisk/scratch/aileenluo/results/run167/pred_amp_object_scan478_crop104_datanorm.npy")
ph_200M_finetune = np.load("/mnt/localdisk/scratch/aileenluo/results/run167/pred_ph_object_scan478_crop104_datanorm.npy")

## Phase reconstruction plots

In [ ]:
# Reference long-exposure LSQML, ePIE, PtychoFM, LSQML

vmin0 = np.mean(ph_gt_lsqml) - (3 * np.std(ph_gt_lsqml))
vmax0 = np.mean(ph_gt_lsqml) + (3 * np.std(ph_gt_lsqml))
vmin1 = np.mean(ph_fly_epie) - (3 * np.std(ph_fly_epie))
vmax1 = np.mean(ph_fly_epie) + (3 * np.std(ph_fly_epie))
vmin2 = np.mean(ph_43M_finetune) - (3 * np.std(ph_43M_finetune))
vmax2 = np.mean(ph_43M_finetune) + (3 * np.std(ph_43M_finetune))
vmin3 = np.mean(ph_fly_lsqml) - (3 * np.std(ph_fly_lsqml))
vmax3 = np.mean(ph_fly_lsqml) + (3 * np.std(ph_fly_lsqml))
print(vmin0, vmax0, vmin1, vmax1, vmin2, vmax2, vmin3, vmax3)

f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2, dpi=300)

gt0 = ax[0, 0].imshow(ph_gt_lsqml, interpolation='none', vmin=vmin0, vmax=vmax0, cmap=cmc.grayC)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT phase')
ax[0, 0].set_xticks([])
ax[0, 0].set_yticks([])
scalebar = AnchoredSizeBar(ax[0, 0].transData, 299, "$2 \mu m$", "lower right", pad=0.2, color="#E8E8E8", frameon=False, size_vertical=5)
ax[0, 0].add_artist(scalebar)

epie1 = ax[0, 1].imshow(ph_fly_epie, interpolation='none', vmin=vmin1, vmax=vmax1, cmap=cmc.grayC)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(epie1, cax=cax, orientation='vertical')
ax[0, 1].set_title('ePIE')
ax[0, 1].set_xticks([])
ax[0, 1].set_yticks([])

ptychofm2 = ax[1, 0].imshow(ph_43M_finetune, interpolation='none', origin='lower', vmin=vmin2, vmax=vmax2, cmap=cmc.grayC)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(ptychofm2, cax=cax, orientation='vertical')
ax[1, 0].set_title('PtychoFM')
ax[1, 0].set_xticks([])
ax[1, 0].set_yticks([])

lsqml3 = ax[1, 1].imshow(ph_fly_lsqml, interpolation='none', vmin=vmin3, vmax=vmax3, cmap=cmc.grayC)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(lsqml3, cax=cax, orientation='vertical')
ax[1, 1].set_title('LSQML')
ax[1, 1].set_xticks([])
ax[1, 1].set_yticks([])

#f.savefig("fig2.svg", transparent=True, bbox_inches="tight", dpi=300)

In [ ]:
# rPIE, DM, RAAR

vmin1 = np.mean(ph_fly_rpie) - (3 * np.std(ph_fly_rpie))
vmax1 = np.mean(ph_fly_rpie) + (3 * np.std(ph_fly_rpie))
vmin2 = np.mean(ph_fly_dm) - (3 * np.std(ph_fly_dm))
vmax2 = np.mean(ph_fly_dm) + (3 * np.std(ph_fly_dm))
vmin3 = np.mean(ph_fly_raar) - (3 * np.std(ph_fly_raar))
vmax3 = np.mean(ph_fly_raar) + (3 * np.std(ph_fly_raar))
print(vmin1, vmax1, vmin2, vmax2, vmin3, vmax3)

f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2, dpi=300)

gt0 = ax[0, 0].imshow(ph_gt_lsqml, interpolation='none', vmin=vmin0, vmax=vmax0, cmap=cmc.grayC)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT phase')
ax[0, 0].set_xticks([])
ax[0, 0].set_yticks([])
scalebar = AnchoredSizeBar(ax[0, 0].transData, 299, "$2 \mu m$", "lower right", pad=0.2, color="#E8E8E8", frameon=False, size_vertical=5)
ax[0, 0].add_artist(scalebar)
ax[0, 0].add_patch(
    plt.Rectangle(
        (520 - 0.5, 420 - 0.5),
        1180 - 520,
        790 - 420,
        fill=False,
        edgecolor="#0072B2",
        linewidth=1.5,
    )
)

epie1 = ax[0, 1].imshow(ph_fly_rpie, interpolation='none', vmin=vmin1, vmax=vmax1, cmap=cmc.grayC)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(epie1, cax=cax, orientation='vertical')
ax[0, 1].set_title('rPIE')
ax[0, 1].set_xticks([])
ax[0, 1].set_yticks([])

ptychofm2 = ax[1, 0].imshow(ph_fly_dm, interpolation='none', vmin=vmin2, vmax=vmax2, cmap=cmc.grayC)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(ptychofm2, cax=cax, orientation='vertical')
ax[1, 0].set_title('DM')
ax[1, 0].set_xticks([])
ax[1, 0].set_yticks([])

lsqml3 = ax[1, 1].imshow(ph_fly_raar, interpolation='none', vmin=vmin3, vmax=vmax3, cmap=cmc.grayC)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(lsqml3, cax=cax, orientation='vertical')
ax[1, 1].set_title('RAAR')
ax[1, 1].set_xticks([])
ax[1, 1].set_yticks([])

#f.savefig("fig2b.svg", transparent=True, bbox_inches="tight", dpi=300)

In [ ]:
# Zoom-in ePIE, PtychoFM, LSQML

vmin0 = np.mean(ph_gt_lsqml[420:790, 520:1180]) - (3 * np.std(ph_gt_lsqml[420:790, 520:1180]))
vmax0 = np.mean(ph_gt_lsqml[420:790, 520:1180]) + (3 * np.std(ph_gt_lsqml[420:790, 520:1180]))
vmin1 = np.mean(ph_fly_epie[420:790, 520:1180]) - (3 * np.std(ph_fly_epie[420:790, 520:1180]))
vmax1 = np.mean(ph_fly_epie[420:790, 520:1180]) + (3 * np.std(ph_fly_epie[420:790, 520:1180]))
vmin2 = np.mean(np.flipud(ph_43M_finetune)[420:790, 520:1180]) - (3 * np.std(np.flipud(ph_43M_finetune)[420:790, 520:1180]))
vmax2 = np.mean(np.flipud(ph_43M_finetune)[420:790, 520:1180]) + (3 * np.std(np.flipud(ph_43M_finetune)[420:790, 520:1180]))
vmin3 = np.mean(ph_fly_lsqml[420:790, 520:1180]) - (3 * np.std(ph_fly_lsqml[420:790, 520:1180]))
vmax3 = np.mean(ph_fly_lsqml[420:790, 520:1180]) + (3 * np.std(ph_fly_lsqml[420:790, 520:1180]))
print(vmin0, vmax0, vmin1, vmax1, vmin2, vmax2, vmin3, vmax3)

f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2, dpi=300)

gt0 = ax[0, 0].imshow(ph_gt_lsqml[420:790, 520:1180], interpolation='none', vmin=vmin0, vmax=vmax0, cmap=cmc.grayC)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT phase')

epie1 = ax[0, 1].imshow(ph_fly_epie[430:790, 520:1180], interpolation='none', vmin=vmin1, vmax=vmax1, cmap=cmc.grayC)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(epie1, cax=cax, orientation='vertical')
ax[0, 1].set_title('ePIE')

ptychofm2 = ax[1, 0].imshow(np.flipud(ph_43M_finetune)[430:790, 520:1180], interpolation='none', vmin=vmin2, vmax=vmax2, cmap=cmc.grayC)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(ptychofm2, cax=cax, orientation='vertical')
ax[1, 0].set_title('PtychoFM')

lsqml3 = ax[1, 1].imshow(ph_fly_lsqml[430:790, 520:1180], interpolation='none', vmin=vmin3, vmax=vmax3, cmap=cmc.grayC)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(lsqml3, cax=cax, orientation='vertical')
ax[1, 1].set_title('LSQML')

#f.savefig("fig2c.svg", transparent=True, bbox_inches="tight", dpi=300)

In [ ]:
# Zoom-in rPIE, DM, RAAR

vmin1 = np.mean(ph_fly_rpie[420:790, 520:1180]) - (3 * np.std(ph_fly_rpie[420:790, 520:1180]))
vmax1 = np.mean(ph_fly_rpie[420:790, 520:1180]) + (3 * np.std(ph_fly_rpie[420:790, 520:1180]))
vmin2 = np.mean(ph_fly_dm[420:790, 520:1180]) - (3 * np.std(ph_fly_dm[420:790, 520:1180]))
vmax2 = np.mean(ph_fly_dm[420:790, 520:1180]) + (3 * np.std(ph_fly_dm[420:790, 520:1180]))
vmin3 = np.mean(ph_fly_raar[420:790, 520:1180]) - (3 * np.std(ph_fly_raar[420:790, 520:1180]))
vmax3 = np.mean(ph_fly_raar[420:790, 520:1180]) + (3 * np.std(ph_fly_raar[420:790, 520:1180]))
print(vmin1, vmax1, vmin2, vmax2, vmin3, vmax3)

f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2, dpi=300)

gt0 = ax[0, 0].imshow(ph_gt_lsqml[420:790, 520:1180], interpolation='none', vmin=vmin0, vmax=vmax0, cmap=cmc.grayC)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT phase')

epie1 = ax[0, 1].imshow(ph_fly_rpie[430:790, 520:1180], interpolation='none', vmin=vmin1, vmax=vmax1, cmap=cmc.grayC)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(epie1, cax=cax, orientation='vertical')
ax[0, 1].set_title('rPIE')

ptychofm2 = ax[1, 0].imshow(ph_fly_dm[430:790, 520:1180], interpolation='none', vmin=vmin2, vmax=vmax2, cmap=cmc.grayC)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(ptychofm2, cax=cax, orientation='vertical')
ax[1, 0].set_title('DM')

lsqml3 = ax[1, 1].imshow(ph_fly_raar[430:790, 520:1180], interpolation='none', vmin=vmin3, vmax=vmax3, cmap=cmc.grayC)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(lsqml3, cax=cax, orientation='vertical')
ax[1, 1].set_title('RAAR')

#f.savefig("fig2d.svg", transparent=True, bbox_inches="tight", dpi=300)

## Amplitude reconstruction plots

In [ ]:
# ePIE, PtychoFM, LSQML

vmin0 = np.mean(amp_gt_lsqml) - (3 * np.std(amp_gt_lsqml))
vmax0 = np.mean(amp_gt_lsqml) + (3 * np.std(amp_gt_lsqml))
vmin1 = np.mean(amp_fly_epie) - (3 * np.std(amp_fly_epie))
vmax1 = np.mean(amp_fly_epie) + (3 * np.std(amp_fly_epie))
vmin2 = np.mean(amp_43M_finetune) - (3 * np.std(amp_43M_finetune))
vmax2 = np.mean(amp_43M_finetune) + (3 * np.std(amp_43M_finetune))
vmin3 = np.mean(amp_fly_lsqml) - (3 * np.std(amp_fly_lsqml))
vmax3 = np.mean(amp_fly_lsqml) + (3 * np.std(amp_fly_lsqml))
print(vmin0, vmax0, vmin1, vmax1, vmin2, vmax2, vmin3, vmax3)

f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2, dpi=300)

gt0 = ax[0, 0].imshow(amp_gt_lsqml, interpolation='none', vmin=vmin0, vmax=vmax0, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')
ax[0, 0].set_xticks([])
ax[0, 0].set_yticks([])
scalebar = AnchoredSizeBar(ax[0, 0].transData, 299, "$2 \mu m$", "lower right", pad=0.2, color="#E8E8E8", frameon=False, size_vertical=5)
ax[0, 0].add_artist(scalebar)

epie1 = ax[0, 1].imshow(amp_fly_epie, interpolation='none', vmin=vmin1, vmax=vmax1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(epie1, cax=cax, orientation='vertical')
ax[0, 1].set_title('ePIE')
ax[0, 1].set_xticks([])
ax[0, 1].set_yticks([])

ptychofm2 = ax[1, 0].imshow(amp_43M_finetune, interpolation='none', origin='lower', vmin=vmin2, vmax=vmax2, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(ptychofm2, cax=cax, orientation='vertical')
ax[1, 0].set_title('PtychoFM')
ax[1, 0].set_xticks([])
ax[1, 0].set_yticks([])

lsqml3 = ax[1, 1].imshow(amp_fly_lsqml, interpolation='none', vmin=vmin3, vmax=vmax3, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(lsqml3, cax=cax, orientation='vertical')
ax[1, 1].set_title('LSQML')
ax[1, 1].set_xticks([])
ax[1, 1].set_yticks([])

#f.savefig("fig2a_amp.svg", transparent=True, bbox_inches="tight", dpi=300)

In [ ]:
# rPIE, DM, RAAR

vmin1 = np.mean(amp_fly_rpie) - (3 * np.std(amp_fly_rpie))
vmax1 = np.mean(amp_fly_rpie) + (3 * np.std(amp_fly_rpie))
vmin2 = np.mean(amp_fly_dm) - (3 * np.std(amp_fly_dm))
vmax2 = np.mean(amp_fly_dm) + (3 * np.std(amp_fly_dm))
vmin3 = np.mean(amp_fly_raar) - (3 * np.std(amp_fly_raar))
vmax3 = np.mean(amp_fly_raar) + (3 * np.std(amp_fly_raar))
print(vmin1, vmax1, vmin2, vmax2, vmin3, vmax3)

f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2, dpi=300)

gt0 = ax[0, 0].imshow(amp_gt_lsqml, interpolation='none', vmin=vmin0, vmax=vmax0, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')
ax[0, 0].set_xticks([])
ax[0, 0].set_yticks([])
scalebar = AnchoredSizeBar(ax[0, 0].transData, 299, "$2 \mu m$", "lower right", pad=0.2, color="#E8E8E8", frameon=False, size_vertical=5)
ax[0, 0].add_artist(scalebar)
ax[0, 0].add_patch(
    plt.Rectangle(
        (520 - 0.5, 420 - 0.5),
        1180 - 520,
        790 - 420,
        fill=False,
        edgecolor="#8F2D3C",
        linewidth=1.5,
    )
)

epie1 = ax[0, 1].imshow(amp_fly_rpie, interpolation='none', vmin=vmin1, vmax=vmax1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(epie1, cax=cax, orientation='vertical')
ax[0, 1].set_title('rPIE')
ax[0, 1].set_xticks([])
ax[0, 1].set_yticks([])

ptychofm2 = ax[1, 0].imshow(amp_fly_dm, interpolation='none', vmin=vmin2, vmax=vmax2, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(ptychofm2, cax=cax, orientation='vertical')
ax[1, 0].set_title('DM')
ax[1, 0].set_xticks([])
ax[1, 0].set_yticks([])

lsqml3 = ax[1, 1].imshow(amp_fly_raar, interpolation='none', vmin=vmin3, vmax=vmax3, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(lsqml3, cax=cax, orientation='vertical')
ax[1, 1].set_title('RAAR')
ax[1, 1].set_xticks([])
ax[1, 1].set_yticks([])

#f.savefig("fig2b_amp.svg", transparent=True, bbox_inches="tight", dpi=300)

In [ ]:
# Zoom-in ePIE, PtychoFM, LSQML

vmin0 = np.mean(amp_gt_lsqml[420:790, 520:1180]) - (3 * np.std(amp_gt_lsqml[420:790, 520:1180]))
vmax0 = np.mean(amp_gt_lsqml[420:790, 520:1180]) + (3 * np.std(amp_gt_lsqml[420:790, 520:1180]))
vmin1 = np.mean(amp_fly_epie[420:790, 520:1180]) - (3 * np.std(amp_fly_epie[420:790, 520:1180]))
vmax1 = np.mean(amp_fly_epie[420:790, 520:1180]) + (3 * np.std(amp_fly_epie[420:790, 520:1180]))
vmin2 = np.mean(np.flipud(amp_43M_finetune)[420:790, 520:1180]) - (3 * np.std(np.flipud(amp_43M_finetune)[420:790, 520:1180]))
vmax2 = np.mean(np.flipud(amp_43M_finetune)[420:790, 520:1180]) + (3 * np.std(np.flipud(amp_43M_finetune)[420:790, 520:1180]))
vmin3 = np.mean(amp_fly_lsqml[420:790, 520:1180]) - (3 * np.std(amp_fly_lsqml[420:790, 520:1180]))
vmax3 = np.mean(amp_fly_lsqml[420:790, 520:1180]) + (3 * np.std(amp_fly_lsqml[420:790, 520:1180]))
print(vmin0, vmax0, vmin1, vmax1, vmin2, vmax2, vmin3, vmax3)

f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2, dpi=300)

gt0 = ax[0, 0].imshow(amp_gt_lsqml[420:790, 520:1180], interpolation='none', vmin=vmin0, vmax=vmax0, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')

epie1 = ax[0, 1].imshow(amp_fly_epie[430:790, 520:1180], interpolation='none', vmin=vmin1, vmax=vmax1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(epie1, cax=cax, orientation='vertical')
ax[0, 1].set_title('ePIE')

ptychofm2 = ax[1, 0].imshow(np.flipud(amp_43M_finetune)[430:790, 520:1180], interpolation='none', vmin=vmin2, vmax=vmax2, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(ptychofm2, cax=cax, orientation='vertical')
ax[1, 0].set_title('PtychoFM')

lsqml3 = ax[1, 1].imshow(amp_fly_lsqml[430:790, 520:1180], interpolation='none', vmin=vmin3, vmax=vmax3, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(lsqml3, cax=cax, orientation='vertical')
ax[1, 1].set_title('LSQML')

#f.savefig("fig2c_amp.svg", transparent=True, bbox_inches="tight", dpi=300)

In [ ]:
# Zoom-in rPIE, DM, RAAR

vmin1 = np.mean(amp_fly_rpie[420:790, 520:1180]) - (3 * np.std(amp_fly_rpie[420:790, 520:1180]))
vmax1 = np.mean(amp_fly_rpie[420:790, 520:1180]) + (3 * np.std(amp_fly_rpie[420:790, 520:1180]))
vmin2 = np.mean(amp_fly_dm[420:790, 520:1180]) - (3 * np.std(amp_fly_dm[420:790, 520:1180]))
vmax2 = np.mean(amp_fly_dm[420:790, 520:1180]) + (3 * np.std(amp_fly_dm[420:790, 520:1180]))
vmin3 = np.mean(amp_fly_raar[420:790, 520:1180]) - (3 * np.std(amp_fly_raar[420:790, 520:1180]))
vmax3 = np.mean(amp_fly_raar[420:790, 520:1180]) + (3 * np.std(amp_fly_raar[420:790, 520:1180]))
print(vmin1, vmax1, vmin2, vmax2, vmin3, vmax3)

f, ax = plt.subplots(figsize=(9, 8), ncols=2, nrows=2, dpi=300)

gt0 = ax[0, 0].imshow(amp_gt_lsqml[420:790, 520:1180], interpolation='none', vmin=vmin0, vmax=vmax0, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(gt0, cax=cax, orientation='vertical')
ax[0, 0].set_title('GT amplitude')

epie1 = ax[0, 1].imshow(amp_fly_rpie[430:790, 520:1180], interpolation='none', vmin=vmin1, vmax=vmax1, cmap=cmc.oslo)
divider = make_axes_locatable(ax[0, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(epie1, cax=cax, orientation='vertical')
ax[0, 1].set_title('rPIE')

ptychofm2 = ax[1, 0].imshow(amp_fly_dm[430:790, 520:1180], interpolation='none', vmin=vmin2, vmax=vmax2, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 0])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(ptychofm2, cax=cax, orientation='vertical')
ax[1, 0].set_title('DM')

lsqml3 = ax[1, 1].imshow(amp_fly_raar[430:790, 520:1180], interpolation='none', vmin=vmin3, vmax=vmax3, cmap=cmc.oslo)
divider = make_axes_locatable(ax[1, 1])
cax = divider.append_axes('right', size='5%', pad=0.05)
f.colorbar(lsqml3, cax=cax, orientation='vertical')
ax[1, 1].set_title('RAAR')

#f.savefig("fig2d_amp.svg", transparent=True, bbox_inches="tight", dpi=300)

## Metrics

In [ ]:
# Contrast-normalized fidelity, PCC, and histogram entropy for amplitude and phase
import pandas as pd
from scipy.stats import pearsonr


def _histogram_entropy_within_robust_range(values, lower, upper, n_bins=256):
    values = np.asarray(values)
    finite_values = values[np.isfinite(values)]
    in_range = finite_values[
        (finite_values >= lower) & (finite_values <= upper)
    ]
    if in_range.size == 0 or upper <= lower:
        return 0.0, 0.0

    counts, _ = np.histogram(
        in_range,
        bins=n_bins,
        range=(lower, upper),
    )
    probabilities = counts[counts > 0] / counts.sum()
    entropy_bits = float(
        -np.sum(probabilities * np.log2(probabilities))
    )
    normalized_entropy = entropy_bits / np.log2(n_bins)
    return entropy_bits, float(normalized_entropy)


def _contrast_normalized_metrics(
    image,
    reference,
    *,
    lower_percentile=1,
    upper_percentile=99,
    entropy_bins=256,
):
    image_values = np.asarray(image).ravel()
    reference_values = np.asarray(reference).ravel()
    valid = np.isfinite(image_values) & np.isfinite(reference_values)
    if valid.sum() < 2:
        raise ValueError('Image/reference pair has fewer than two finite pixels')

    image_values = image_values[valid].astype(np.float64, copy=False)
    reference_values = reference_values[valid].astype(np.float64, copy=False)
    image_p01, image_p99 = np.percentile(
        image_values, [lower_percentile, upper_percentile]
    )
    reference_p01, reference_p99 = np.percentile(
        reference_values, [lower_percentile, upper_percentile]
    )
    image_range = float(image_p99 - image_p01)
    reference_range = float(reference_p99 - reference_p01)
    if image_range <= 0 or reference_range <= 0:
        raise ValueError('The 1st–99th percentile range must be positive')

    # Each image is expressed in units of its own robust contrast. The mean
    # residual is the additive shift that minimizes the subsequent MSE.
    scaled_image = image_values / image_range
    scaled_reference = reference_values / reference_range
    mean_shift = float(np.mean(scaled_reference - scaled_image))
    residual = scaled_image + mean_shift - scaled_reference
    mse = float(np.mean(residual**2))
    entropy_bits, normalized_entropy = (
        _histogram_entropy_within_robust_range(
            image_values,
            image_p01,
            image_p99,
            n_bins=entropy_bins,
        )
    )

    return {
        'p01': float(image_p01),
        'p99': float(image_p99),
        'robust_range': image_range,
        'mean_shift': mean_shift,
        'contrast_normalized_mse': mse,
        'contrast_normalized_rmse': float(np.sqrt(mse)),
        'pcc': float(pearsonr(image_values, reference_values).statistic),
        'entropy_bits': entropy_bits,
        'entropy_normalized': normalized_entropy,
    }


def _compact_modality_metrics(full_dataframe, modality):
    prefix = f'{modality}_'
    compact = pd.DataFrame(index=full_dataframe.index)
    compact['percentiles_1_99'] = list(zip(
        full_dataframe[f'{prefix}p01'],
        full_dataframe[f'{prefix}p99'],
    ))
    compact['robust_range'] = full_dataframe[f'{prefix}robust_range']
    compact['mean_shift'] = full_dataframe[f'{prefix}mean_shift']
    compact['contrast_normalized_mse'] = full_dataframe[
        f'{prefix}contrast_normalized_mse'
    ]
    compact['contrast_normalized_rmse'] = full_dataframe[
        f'{prefix}contrast_normalized_rmse'
    ]
    compact['pcc'] = full_dataframe[f'{prefix}pcc']
    return compact


def _build_contrast_metric_dataframes(metric_images, reference_name):
    reference = metric_images[reference_name]
    records = []
    for name, images in metric_images.items():
        amplitude_result = _contrast_normalized_metrics(
            images['amplitude'],
            reference['amplitude'],
        )
        phase_result = _contrast_normalized_metrics(
            images['phase'],
            reference['phase'],
        )
        record = {
            'method': name,
            'is_reference': name == reference_name,
        }
        record.update({
            f'amplitude_{key}': value
            for key, value in amplitude_result.items()
        })
        record.update({
            f'phase_{key}': value
            for key, value in phase_result.items()
        })
        records.append(record)

    full_dataframe = (
        pd.DataFrame.from_records(records)
        .set_index('method')
        .rename_axis('reconstruction')
    )
    amplitude_dataframe = _compact_modality_metrics(
        full_dataframe, 'amplitude'
    )
    phase_dataframe = _compact_modality_metrics(full_dataframe, 'phase')
    return full_dataframe, amplitude_dataframe, phase_dataframe


raw_contrast_metric_images = {
    'gt_lsqml': (amp_gt_lsqml, ph_gt_lsqml),
    'gt_epie': (amp_gt_epie, ph_gt_epie),
    'fly_lsqml': (amp_fly_lsqml, ph_fly_lsqml),
    'fly_epie': (amp_fly_epie, ph_fly_epie),
    'fly_rpie': (amp_fly_rpie, ph_fly_rpie),
    'fly_dm': (amp_fly_dm, ph_fly_dm),
    'fly_raar': (amp_fly_raar, ph_fly_raar),
    '43M_finetune': (amp_43M_finetune, ph_43M_finetune),
    '43M_finetune_datanorm': (
        amp_43M_finetune_datanorm,
        ph_43M_finetune_datanorm,
    ),
    '200M_finetune': (amp_200M_finetune, ph_200M_finetune),
    '200M_finetune_datanorm': (
        amp_200M_finetune_datanorm,
        ph_200M_finetune_datanorm,
    ),
    '43M_scratch': (amp_43M_scratch, ph_43M_scratch),
    '43M_scratch_datanorm': (
        amp_43M_scratch_datanorm,
        ph_43M_scratch_datanorm,
    ),
    '200M_scratch': (amp_200M_scratch, ph_200M_scratch),
    '200M_scratch_datanorm': (
        amp_200M_scratch_datanorm,
        ph_200M_scratch_datanorm,
    ),
}

contrast_metric_images = {}
for name, (amplitude_image, phase_image) in raw_contrast_metric_images.items():
    if amplitude_image.shape != phase_image.shape:
        raise ValueError(
            f'Amplitude/phase shape mismatch for {name}: '
            f'{amplitude_image.shape} versus {phase_image.shape}'
        )
    if '43M' in name or '200M' in name:
        amplitude_image = np.flipud(amplitude_image)
        phase_image = np.flipud(phase_image)
    contrast_metric_images[name] = {
        'amplitude': amplitude_image,
        'phase': phase_image,
    }

contrast_metric_reference_name = 'gt_lsqml'
(
    contrast_normalized_metrics_df,
    amplitude_contrast_metrics_df,
    phase_contrast_metrics_df,
) = _build_contrast_metric_dataframes(
    contrast_metric_images,
    contrast_metric_reference_name,
)

print(
    'Full-image contrast-normalized metrics relative to '
    f'{contrast_metric_reference_name}; each reconstruction uses its own '
    '1st–99th percentile range; entropy uses 256 bins within that range.'
)
display(contrast_normalized_metrics_df)
print('Amplitude metrics')
display(amplitude_contrast_metrics_df)
print('Phase metrics')
display(phase_contrast_metrics_df)

In [ ]:
# Placeholder ROI for contrast-normalized metrics; update these slices as needed.
contrast_metric_row_slice = slice(430, 790)
contrast_metric_column_slice = slice(520, 1180)

# Show the selected ROI on the phase reference used for these metrics.
phase_reference = contrast_metric_images[contrast_metric_reference_name]["phase"]
row_start, row_stop, _ = contrast_metric_row_slice.indices(phase_reference.shape[0])
column_start, column_stop, _ = contrast_metric_column_slice.indices(
    phase_reference.shape[1]
)

slice_location_fig, slice_location_ax = plt.subplots(figsize=(8, 8))
slice_location_image = slice_location_ax.imshow(phase_reference, cmap=cmc.vik)
slice_location_ax.add_patch(
    plt.Rectangle(
        (column_start - 0.5, row_start - 0.5),
        column_stop - column_start,
        row_stop - row_start,
        fill=False,
        edgecolor="red",
        linewidth=2.5,
    )
)
slice_location_ax.set_title(
    f"Phase metric slice on {contrast_metric_reference_name}: "
    f"rows {row_start}:{row_stop}, columns {column_start}:{column_stop}"
)
slice_location_ax.set_xlabel("Column (pixels)")
slice_location_ax.set_ylabel("Row (pixels)")
slice_location_fig.colorbar(
    slice_location_image, ax=slice_location_ax, label="Phase (rad)"
)
slice_location_fig.tight_layout()
plt.show()

sliced_contrast_metric_images = {
    name: {
        modality: image[
            contrast_metric_row_slice,
            contrast_metric_column_slice,
        ]
        for modality, image in images.items()
    }
    for name, images in contrast_metric_images.items()
}

(
    sliced_contrast_normalized_metrics_df,
    sliced_amplitude_contrast_metrics_df,
    sliced_phase_contrast_metrics_df,
) = _build_contrast_metric_dataframes(
    sliced_contrast_metric_images,
    contrast_metric_reference_name,
)

print(
    'Sliced contrast-normalized metrics relative to '
    f'{contrast_metric_reference_name}; rows={contrast_metric_row_slice}, '
    f'columns={contrast_metric_column_slice}.'
)
display(sliced_contrast_normalized_metrics_df)
print('Sliced amplitude metrics')
display(sliced_amplitude_contrast_metrics_df)
print('Sliced phase metrics')
display(sliced_phase_contrast_metrics_df)

def _values_normalized_to_robust_range(image):
    values = np.asarray(image).ravel()
    values = values[np.isfinite(values)].astype(np.float64, copy=False)
    p01, p99 = np.percentile(values, [1, 99])
    robust_range = p99 - p01
    if robust_range <= 0:
        raise ValueError('The 1st–99th percentile range must be positive')
    in_range = values[(values >= p01) & (values <= p99)]
    return (in_range - p01) / robust_range


def _plot_sliced_robust_histograms(modality, metrics_dataframe):
    method_names = list(sliced_contrast_metric_images)
    n_columns = 4
    n_rows = int(np.ceil(len(method_names) / n_columns))
    fig, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(18, 3.8 * n_rows),
        sharex=True,
    )
    axes = np.atleast_1d(axes).ravel()
    reference_values = _values_normalized_to_robust_range(
        sliced_contrast_metric_images[contrast_metric_reference_name][modality]
    )

    for method_index, name in enumerate(method_names):
        ax = axes[method_index]
        values = _values_normalized_to_robust_range(
            sliced_contrast_metric_images[name][modality]
        )
        color = plt.get_cmap('tab20')(
            method_index / max(len(method_names) - 1, 1)
        )
        if name != contrast_metric_reference_name:
            ax.hist(
                reference_values,
                bins=256,
                range=(0, 1),
                density=True,
                histtype='step',
                color='black',
                linewidth=1.0,
                alpha=0.65,
                label=contrast_metric_reference_name,
            )
        ax.hist(
            values,
            bins=256,
            range=(0, 1),
            density=True,
            histtype='step',
            color=color,
            linewidth=1.3,
            label=name,
        )
        entropy_bits = sliced_contrast_normalized_metrics_df.loc[
            name, f'{modality}_entropy_bits'
        ]
        ax.set_title(f'{name}\nH = {entropy_bits:.3f} bits', fontsize=10)
        ax.set_xlim(0, 1)
        ax.set_xlabel('Normalized 1st–99th percentile range')
        ax.set_ylabel('Probability density')
        ax.grid(alpha=0.15)
        ax.legend(fontsize=7, loc='best')

    for ax in axes[len(method_names):]:
        ax.set_visible(False)

    fig.suptitle(
        f'Sliced {modality} histograms; reference overlay = '
        f'{contrast_metric_reference_name}',
        fontsize=15,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


sliced_amplitude_histogram_fig = _plot_sliced_robust_histograms(
    'amplitude', sliced_amplitude_contrast_metrics_df
)
sliced_phase_histogram_fig = _plot_sliced_robust_histograms(
    'phase', sliced_phase_contrast_metrics_df
)
plt.show()